# Chapter 6 — Accelerations and proper-motion anomalies

The HGCA and Gaia extracts download automatically from the tutorial data release, with SHA-256 checks.

**Goal.** Understand what an acceleration measures when an orbit is only partly observed, inspect the DR3
acceleration sample, and distinguish a short-baseline proper motion from a long-baseline positional difference.
An acceleration alone does not give a unique companion mass or period.

We use the DR3 acceleration processing described by Halbwachs et al. (2023,
https://arxiv.org/abs/2206.05726), the NSS counts of Gaia Collaboration, Arenou et al. (2023,
https://arxiv.org/abs/2206.05595), and the proper-motion comparison developed in the HGCA by Brandt (2021,
https://arxiv.org/abs/2105.11662).

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts, attached to a GitHub release of the tutorial. MANIFEST.json lists, for each file,
# its origin (SQL query or URL) and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/gaia-nss-tutorial/releases/download/data-v0.3/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)
from scipy.stats import chi2


## 6.1 A small piece of an orbit

For a circular photocentre orbit projected onto the sky, take
$\boldsymbol{x}(t)=a_0(\cos(\omega t+\phi),\cos i\,\sin(\omega t+\phi))$ with $\omega=2\pi/P$.
Near $t=0$ it is a position, a proper motion, an acceleration, and a jerk:

$$\boldsymbol{x}(t)=\boldsymbol{x}_0+\boldsymbol{\mu}_0 t+
\tfrac12\boldsymbol{g}_0t^2+\tfrac16\boldsymbol{j}_0t^3+\cdots,$$

where $\boldsymbol{g}_0=-\omega^2\boldsymbol{x}_0$ and
$\boldsymbol{j}_0=-\omega^2\boldsymbol{\mu}_0$. This is a local Taylor expansion, not a reproduction of
Gaia's fitting basis or its pipeline. We have omitted parallax to isolate the orbital curvature.

In [ ]:
a0, P, cosi, phase = 1.0, 30.0, 0.5, 0.7    # mas, yr, cos(inclination), radians
T = 34 / 12                                    # observation span [yr]
t = np.linspace(-T/2, T/2, 400)
w = 2*np.pi/P
x0 = a0*np.array([np.cos(phase), cosi*np.sin(phase)])
mu0 = a0*w*np.array([-np.sin(phase), cosi*np.cos(phase)])
g0, j0 = -w*w*x0, -w*w*mu0                       # mas/yr^2 and mas/yr^3
exact = a0*np.column_stack([np.cos(w*t+phase), cosi*np.sin(w*t+phase)])
linear = x0 + t[:, None]*mu0                    # [:, None] gives one time column for both sky coordinates
quadratic = linear + t[:, None]**2*g0/2
cubic = quadratic + t[:, None]**3*j0/6
err2 = np.linalg.norm(exact-quadratic, axis=1)    # angular distance between the two sky positions [mas]
err3 = np.linalg.norm(exact-cubic, axis=1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for model, label in [(exact, 'orbit'), (quadratic, 'constant acceleration'), (cubic, 'acceleration + jerk')]:
    ax[0].plot(t, (model-linear)[:, 0], label=label)
ax[0].set(xlabel='time from midpoint [yr]', ylabel='east offset after removing tangent [mas]')
ax[0].legend(fontsize=8)
ax[1].semilogy(t, err2, label='quadratic error')
ax[1].semilogy(t, err3, label='cubic error')
ax[1].set(xlabel='time from midpoint [yr]', ylabel='sky-position error [mas]')
ax[1].legend(); plt.tight_layout(); plt.show()
# Analytic Taylor remainder bounds: the derivative norms are at most a0*w^3 and a0*w^4.
# Allow 1e-12 mas for floating-point subtraction; no tolerance is chosen from a measured result.
assert err2.max() <= a0*(w*T/2)**3/6 + 1e-12
assert err3.max() <= a0*(w*T/2)**4/24 + 1e-12
assert err3.max() < err2.max()
print(f'Maximum position errors: quadratic {err2.max():.5f}, cubic {err3.max():.5f} mas')

**Figure 6.1.** A simulated 30-year circular photocentre orbit, $a_0=1$ mas and $i=60^\circ$, sampled over
34 months. Left: east displacement after subtracting the tangent, against time in years; blue is the orbit,
orange its quadratic expansion, and green its cubic expansion. Right: angular error in mas of the quadratic
(blue) and cubic (orange) approximations. Neither panel contains measurement noise.

Over this short arc, acceleration describes most of the curvature and jerk improves the approximation.
The remainder checks compare these approximations with the exact orbit. Extrapolating the polynomial around a
full orbit is a different problem: a good local description does not determine the orbital period.

## 6.2 What the DR3 acceleration table contains

`Acceleration7` adds two accelerations to the five astrometric parameters; `Acceleration9` also includes their
two time derivatives. The public data model defines `accel_ra` and `accel_dec` in mas/yr², and `deriv_accel_*`
in mas/yr³. See the [ESA column definitions](https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_non--single_stars_tables/ssec_dm_nss_acceleration_astro.html).

We plot the angular acceleration amplitude $g=\sqrt{g_{\alpha*}^2+g_\delta^2}$.
This is an amplitude, not a detection significance. The current extract does not include the acceleration
correlation matrix, so we do not manufacture a two-dimensional significance from marginal errors.

In [ ]:
acc = load('nss_acceleration')
counts = acc.nss_solution_type.value_counts().reindex(['Acceleration7', 'Acceleration9'])
amp = np.hypot(acc.accel_ra, acc.accel_dec)        # stable evaluation of sqrt(g_RA^2 + g_Dec^2)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(['7 parameters', '9 parameters'], counts, color=['C0', 'C1'])
ax[0].set(ylabel='published solutions')
for label, color in [('Acceleration7', 'C0'), ('Acceleration9', 'C1')]:
    selected = acc.nss_solution_type == label
    ax[1].hist(amp[selected], bins=np.geomspace(1e-3, 1e2, 90), histtype='step', color=color, label=label)
ax[1].set(xscale='log', xlabel='angular acceleration amplitude [mas/yr²]', ylabel='solutions per logarithmic bin')
ax[1].legend(); plt.tight_layout(); plt.show()
# Published counts: Arenou et al. (2023), Table 1; exact agreement is required.
assert counts.to_dict() == {'Acceleration7': 246947, 'Acceleration9': 91268}
assert len(acc) == 338215
print(counts.to_string())

**Figure 6.2.** All 338,215 published DR3 acceleration solutions in the shipped extract. Left: counts of
seven-parameter (blue) and nine-parameter (orange) solutions. Right: their angular acceleration amplitudes,
in mas/yr², with counts in logarithmic bins and the same colours. The assertion reproduces the published counts.

The two groups overlap in acceleration amplitude. Their relative numbers describe the released models, not the
fraction of all stars with long-period companions: the input selection and model-acceptance cascade matter
(Halbwachs et al. 2023). A fitted acceleration also requires validation before a companion interpretation;
perspective effects and other astrometric systematics must be considered.

## 6.3 A proper-motion anomaly is a difference of slopes

Let two position measurements be separated by $\Delta t$. Their difference gives an average proper motion
$\mu_{HG}=[x(t_G)-x(t_H)]/\Delta t$. For constant acceleration, and taking $t_G=0$,

$$\mu_G-\mu_{HG}=\tfrac12 g\Delta t.$$

This identity is a useful limit, not a general conversion of HGCA anomalies into instantaneous accelerations.
Real catalogue proper motions average finite observing windows. The HGCA cross-calibrates Hipparcos and Gaia
proper motions and the long-baseline positional difference, including uncertainty calibration (Brandt 2021).
Our example uses an idealised 24.75-year separation and independent errors in the two proper-motion estimates.

In [ ]:
dt, x_ref, mu, g = 24.75, 0.0, 1.0, 0.02     # yr, mas, mas/yr, mas/yr²
position = lambda time: x_ref + mu*time + 0.5*g*time**2
mu_hg = (position(0)-position(-dt))/dt
anomaly = mu-mu_hg
rng = np.random.default_rng(6)
# Toy covariance of a two-component proper-motion difference [mas²/yr²].
# For independent estimates C_delta = C_G + C_HG; the off-diagonal element correlates the sky axes.
C_delta = np.array([[0.05**2 + 0.02**2, 0.4*0.05*0.07],
                    [0.4*0.05*0.07, 0.07**2 + 0.02**2]])
N = 20000
noise = rng.multivariate_normal([0, 0], C_delta, size=N)  # zero-acceleration null sample
# Mahalanobis distance: d^2 = delta_mu.T @ inv(C_delta) @ delta_mu, computed without forming the inverse.
d2 = np.sum(noise*np.linalg.solve(C_delta, noise.T).T, axis=1)
threshold = chi2.ppf(0.99, df=2)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
tt = np.linspace(-dt, 2, 300)
ax[0].plot(tt, position(tt), label='constant acceleration')
ax[0].plot(tt, x_ref + mu_hg*tt, '--', label='long-baseline slope')
ax[0].plot(np.linspace(-5, 2, 50), x_ref+mu*np.linspace(-5, 2, 50), ':', label='current tangent')
ax[0].set(xlabel='time from recent epoch [yr]', ylabel='one sky coordinate [mas]')
ax[0].legend(fontsize=8)
ax[1].hist(d2, bins=np.linspace(0, 15, 60), density=True, histtype='step', label='simulated null anomalies')
x = np.linspace(0, 15, 300)
ax[1].plot(x, chi2.pdf(x, df=2), 'k--', label='chi-square, 2 degrees of freedom')
ax[1].axvline(threshold, color='C3', label='99% threshold')
ax[1].set(xlabel='squared significance d²', ylabel='probability density')
ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()
# Analytic identity, absolute tolerance 1e-12 mas/yr.
assert abs(anomaly-0.5*g*dt) < 1e-12
# Ensemble calibration: E[chi²_2] = 2 and a 1% upper tail, each within 3 standard errors.
assert abs(d2.mean()-2) < 3*np.sqrt(4/N)
assert abs(np.mean(d2 > threshold)-0.01) < 3*np.sqrt(0.01*0.99/N)
print(f'Constant-acceleration anomaly: {anomaly:.4f} mas/yr; null mean d²: {d2.mean():.3f}')

**Figure 6.3.** Left: a simulated constant-acceleration trajectory (blue), the slope from two positions
24.75 years apart (orange dashed), and the recent tangent (green dotted), in mas against years.
Right: squared significances of 20,000 simulated zero-acceleration proper-motion differences (blue), compared
with the analytic two-degree-of-freedom chi-square density (black dashed); the red line marks its 99% quantile.

The first panel explains the difference of slopes. The second shows why the covariance, not just the length of
the proper-motion difference, sets the false-positive rate. If the two estimates share errors, the covariance of
their difference also needs the two cross-covariance terms; simply adding their covariance matrices would be wrong.



## 6.4 The measured Hipparcos–Gaia anomaly

Brandt (2021, https://arxiv.org/abs/2105.11662, Sections 4 and 8) supplies three proper motions in the
EDR3 frame: Hipparcos, Gaia, and the position difference divided by the source-specific epoch difference.
Use its **calibrated** uncertainties as supplied; do not multiply them by the paper's error-inflation factor again.
RA proper motions already include $\cos\delta$. The central RA and Dec epochs differ from one another and
from the nominal mission epochs. Propagating positions to these central epochs removes their same-coordinate
position–proper-motion covariance; we use the catalogue's two-matrix approximation for the Gaia–HG difference.
This is not a general license to ignore cross-covariance in other catalogues.

The cross-calibration and nonlinear sky-projection corrections are already in the HGCA proper motions.
We retain those fields in the extract but do not apply them again. A full orbit fit needs the observing windows
and the conventions in the paper, not an instantaneous acceleration inferred by dividing by 24.75 years.

For each motion, construct $C_{\alpha\delta}=\rho\sigma_\alpha\sigma_\delta$. Then
$D_G^2=(\boldsymbol\mu_G-\boldsymbol\mu_{HG})^T(C_G+C_{HG})^{-1}
(\boldsymbol\mu_G-\boldsymbol\mu_{HG})$. We solve the linear system rather than explicitly invert $C$.
The catalogue's `chisq` uses the **two most precise** motions, which need not be this pair for every star.
We validate against the ten stars printed in the paper's Table 3, for which the Gaia–HG pair reproduces the
published values. The tolerance of 0.051 accounts for rounding that table to one decimal place.


In [ ]:
h = load('hgca_edr3')
assert len(h) == 115346                         # published catalogue size, exact count
assert h.gaia_source_id.dtype.kind in 'iu' and h.gaia_source_id.is_unique
assert h.hip_id.is_unique

def pm_covariance(frame, suffix):
    # Convert to float64 before arithmetic; the catalogue stores most measurements as float32.
    sx = frame['pmra_'+suffix+'_error'].to_numpy(dtype=float)
    sy = frame['pmdec_'+suffix+'_error'].to_numpy(dtype=float)
    rho = frame['pmra_pmdec_'+suffix].to_numpy(dtype=float)
    assert np.all((sx > 0) & (sy > 0) & (np.abs(rho) < 1))
    cov = np.zeros((len(frame), 2, 2))          # one 2-by-2 matrix per star
    cov[:, 0, 0], cov[:, 1, 1] = sx**2, sy**2
    cov[:, 0, 1] = cov[:, 1, 0] = rho*sx*sy
    return cov

delta = h[['pmra_gaia', 'pmdec_gaia']].to_numpy(dtype=float) - h[['pmra_hg', 'pmdec_hg']].to_numpy(dtype=float)
cov = pm_covariance(h, 'gaia') + pm_covariance(h, 'hg')
assert np.all(np.linalg.eigvalsh(cov) > 0)      # each covariance must be positive definite
weighted = np.linalg.solve(cov, delta[..., None])[..., 0]  # batched column-vector solve
h['d2_gaia_hg'] = np.sum(delta*weighted, axis=1)
first = h.set_index('hip_id').loc[np.arange(1, 11)]
published = np.array([3515.5, 5.5, 4.2, 0.2, 1.1, 0.8, 6859.7, 8.6, 3.3, 0.6])
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(first.index, first.d2_gaia_hg, 'o', label='computed Gaia–HG')
ax.semilogy(first.index, published, 'kx', label='Brandt Table 3 (rounded)')
ax.axhline(chi2.isf(0.0027, 2), color='tab:red', ls='--', label='0.27% null tail')
ax.set(xlabel='Hipparcos identifier', ylabel=r'$D_G^2$ (dimensionless)', xticks=np.arange(1, 11))
ax.legend(); plt.tight_layout(); plt.show()
assert np.allclose(first.d2_gaia_hg, published, rtol=0, atol=0.051), 'Table 3, one-decimal rounding'
assert np.allclose(first.chisq, published, rtol=0, atol=0.051)


**Figure 6.4.** Proper-motion differences for HIP 1–10. Blue circles show the computed Gaia–HG squared
significance; black crosses show the rounded values in Brandt (2021), Table 3. The vertical axis is logarithmic
and dimensionless. The dashed red line is the two-degree-of-freedom chi-square threshold for a 0.27% null tail.
HIP 1 and HIP 7 stand well above this line; the other eight fall below it. This reproduces the paper's example,
including its covariance convention. It does not establish that every large anomaly is a companion.

## 6.5 Which DR3 acceleration sources are also in HGCA?

EDR3 and DR3 use the same source identifiers, so we join by the integer ID rather than a sky cone. Check that
both sides have unique IDs and demand a one-to-one merge. This validates the mechanics of the join, not the
physical reliability of every original Hipparcos association. HGCA is a bright-star sample, whereas the DR3
acceleration catalogue has different selection cuts. Compare the measured distributions without treating the
overlap fraction as completeness or dividing an HGCA anomaly by a fixed baseline to estimate a mass.


In [ ]:
assert acc.source_id.is_unique
matched = h.merge(acc[['source_id', 'nss_solution_type', 'accel_ra', 'accel_dec']],
                  left_on='gaia_source_id', right_on='source_id', how='inner', validate='one_to_one')
assert len(matched) > 0 and matched.source_id.is_unique
print(f'{len(matched):,} exact-ID matches out of {len(h):,} HGCA and {len(acc):,} DR3 acceleration entries')
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
bins = np.geomspace(1e-4, 1e6, 71)
for frame, label, colour in [(h, 'all HGCA', '0.5'), (matched, 'HGCA with DR3 acceleration', 'tab:blue')]:
    ax[0].hist(frame.d2_gaia_hg, bins=bins, histtype='step', color=colour, label=label)
ax[0].set(xscale='log', yscale='log', xlabel=r'$D_G^2$ (dimensionless)', ylabel='Stars per bin')
ax[0].legend(fontsize=8)
for kind, colour in [('Acceleration7', 'tab:blue'), ('Acceleration9', 'tab:orange')]:
    subset = matched[matched.nss_solution_type == kind]
    ax[1].scatter(np.sqrt(subset.d2_gaia_hg), np.hypot(subset.accel_ra, subset.accel_dec),
                  s=8, alpha=0.35, color=colour, label=kind)
ax[1].set(xscale='log', yscale='log', xlabel=r'$D_G$ (dimensionless)', ylabel='DR3 acceleration amplitude [mas/yr²]')
ax[1].legend(); plt.tight_layout(); plt.show()


**Figure 6.5.** Left: counts per logarithmic squared-significance bin for all HGCA stars (grey) and those
with an exact-ID DR3 acceleration match (blue); both axes are logarithmic. Right: DR3 angular acceleration
amplitude in mas/yr² against dimensionless Gaia–HG significance, for the matched stars; blue and orange
mark seven- and nine-parameter DR3 solutions. Points are translucent to expose overlap.

The two axes on the right describe different measurements: one is an acceleration fitted over the Gaia
observing span, the other is a difference of slopes across the Hipparcos–Gaia baseline divided by its
uncertainty. Their association is not a conversion law. The unmatched HGCA stars are not a verified single-star
control sample. Brandt explicitly cautions against using the catalogue `chisq` as a population selection function.

**Exercises.** Shorten the period in Figure 6.1 and identify where the acceleration expansion fails.
Repeat Figure 6.4 after setting the off-diagonal covariance to zero. For HIP 1, inspect the four individual
reference epochs and the stored nonlinear correction terms before proposing an orbit model.

No companion mass or real-survey completeness estimate is claimed here.


## References

- Halbwachs et al. (2023), *Gaia Data Release 3. Astrometric binary star processing*, A&A 674, A9: https://arxiv.org/abs/2206.05726
- Gaia Collaboration, Arenou et al. (2023), *Gaia Data Release 3: Stellar multiplicity, a teaser for the hidden treasure*, A&A 674, A34: https://arxiv.org/abs/2206.05595
- Brandt (2021), *The Hipparcos-Gaia Catalog of Accelerations: Gaia EDR3 Edition*, ApJS 254, 42: https://arxiv.org/abs/2105.11662
- ESA, *Gaia DR3 data model: nss_acceleration_astro*: https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_non--single_stars_tables/ssec_dm_nss_acceleration_astro.html